# Phân tích thực nghiệm Gomoku (Caro) 9×9

Notebook này đọc trực tiếp các tệp nhật ký JSONL trong `logs/` do `arena.round_robin` sinh ra để tổng hợp:
1. Bảng kết quả Thắng / Hòa / Thua, điểm chuẩn hóa `score = (wins + 0.5 * draws) / games`, khoảng tin cậy **Wilson 95%**, thời gian trung bình/nước, độ trễ **P95** và khối lượng duyệt (`nodes` / `simulations`).
2. Ma trận đối đầu trực tiếp (Head-to-Head Heatmap) giữa các cấp độ AI.
3. So sánh hai chế độ ngân sách (`compute` mặc định và `time` cùng ngân sách thời gian/nước) nhằm trả lời câu hỏi nghiên cứu **RQ1**.
4. Phân tích lợi thế đi trước (quân `X` so với quân `O`) và phân bố độ dài ván cờ.

In [1]:
from pathlib import Path
import json
import sys
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from arena.round_robin import (
    format_matrix,
    format_summary,
    head_to_head,
    side_advantage,
    summarize,
)

LOG_DIR = ROOT / "logs"
FIG_DIR = ROOT / "notebooks" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

AGENTS = ["random", "heuristic", "alphabeta", "mcts"]

def load_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]

runs = {}
for label, filename in (("compute", "rr_compute_100.jsonl"), ("time", "rr_time_100.jsonl")):
    recs = load_jsonl(LOG_DIR / filename)
    if recs:
        runs[label] = recs

if not runs and (LOG_DIR / "round_robin.jsonl").exists():
    runs["compute"] = load_jsonl(LOG_DIR / "round_robin.jsonl")

summaries = {}
for mode, recs in runs.items():
    summaries[mode] = summarize(recs, agent_names=AGENTS)
    adv = side_advantage(recs)
    print(f"=== Chế độ ngân sách: {mode} ({len(recs)} ván) ===")
    print(format_summary(summaries[mode]))
    print("\nMa trận đối đầu (score | W-D-L):")
    print(format_matrix(head_to_head(recs, AGENTS), AGENTS))
    print(
        f"\nLợi thế đi trước: X thắng {adv['x_wins']}/{adv['games']} ({adv['x_win_rate']:.1%}), "
        f"O thắng {adv['o_wins']}/{adv['games']} ({adv['o_win_rate']:.1%}), "
        f"Hòa {adv['draws']}/{adv['games']} ({adv['draw_rate']:.1%})\n"
    )

=== Chế độ ngân sách: compute (600 ván) ===
Agent         Thắng    Hòa   Thua    Điểm          CI 95%   Thời gian/nước (s)    P95 (s)  Nodes/Sims
-----------------------------------------------------------------------------------------------------
random            0      0    300   0.000    [0.00, 0.01]             0.000012   0.000028         0.0
heuristic       207      2     91   0.693    [0.64, 0.74]             0.001579   0.003748         0.0
alphabeta       291      2      7   0.973    [0.95, 0.99]             0.064807   0.141135         0.0
mcts            100      0    200   0.333    [0.28, 0.39]             0.057916   0.099276       400.0

Ma trận đối đầu (score | W-D-L):
A \ B                   random         heuristic         alphabeta              mcts
------------------------------------------------------------------------------------
random                     ---    0.00 (0-0-100)    0.00 (0-0-100)    0.00 (0-0-100)
heuristic       1.00 (100-0-0)               ---     0.

In [2]:
modes = list(runs.keys())
fig, axes = plt.subplots(1, len(modes), figsize=(6.5 * len(modes), 5.2), squeeze=False, layout="constrained")

for idx, mode in enumerate(modes):
    ax = axes[0, idx]
    h2h = head_to_head(runs[mode], AGENTS)
    n = len(AGENTS)
    grid = np.full((n, n), np.nan)
    for i, a in enumerate(AGENTS):
        for j, b in enumerate(AGENTS):
            if i != j:
                grid[i, j] = h2h[a][b]["score"]

    im = ax.imshow(grid, vmin=0.0, vmax=1.0, cmap="RdYlGn")
    ax.set_xticks(range(n), AGENTS)
    ax.set_yticks(range(n), AGENTS)
    ax.set_xlabel("Đối thủ (cột B)")
    ax.set_ylabel("Tác tử đánh giá (hàng A)")
    ax.set_title(f"Ma trận đối đầu — {mode} ({len(runs[mode])} ván)")

    for i, a in enumerate(AGENTS):
        for j, b in enumerate(AGENTS):
            if i == j:
                ax.text(j, i, "—", ha="center", va="center", fontweight="bold")
            else:
                c = h2h[a][b]
                txt = f"{c['score']:.2f}\n({c['wins']}-{c['draws']}-{c['losses']})"
                ax.text(j, i, txt, ha="center", va="center", fontsize=9)

fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.82, label="Điểm chuẩn hóa (score)")
out_h2h = FIG_DIR / "head_to_head_matrix.png"
fig.savefig(out_h2h, dpi=160, bbox_inches="tight")
print("Đã lưu:", out_h2h)
plt.show()

Đã lưu: C:\Users\Admin\AdvancedAi_for_CARO\notebooks\figures\head_to_head_matrix.png


In [3]:
fig, (ax_score, ax_time) = plt.subplots(1, 2, figsize=(12, 4.6))
x = np.arange(len(AGENTS))
width = 0.36 if len(modes) > 1 else 0.5

for idx, mode in enumerate(modes):
    offset = (idx - (len(modes) - 1) / 2) * width
    s_map = summaries[mode]
    scores = [s_map[a]["score"] for a in AGENTS]
    err_low = [s_map[a]["score"] - s_map[a]["ci_low"] for a in AGENTS]
    err_high = [s_map[a]["ci_high"] - s_map[a]["score"] for a in AGENTS]
    ax_score.bar(x + offset, scores, width, yerr=[err_low, err_high], capsize=4, label=f"mode={mode}")

    means_ms = [s_map[a]["time_per_move"] * 1000.0 for a in AGENTS]
    p95_ms = [s_map[a]["p95_time"] * 1000.0 for a in AGENTS]
    ax_time.bar(x + offset, means_ms, width, label=f"TB ({mode})")
    ax_time.scatter(x + offset, p95_ms, marker="D", zorder=5, label=f"P95 ({mode})")

ax_score.set_xticks(x, AGENTS)
ax_score.set_ylim(0.0, 1.08)
ax_score.set_ylabel("Score = (Thắng + 0.5 × Hòa) / Số ván")
ax_score.set_title("Điểm thi đấu & Khoảng tin cậy Wilson 95%")
ax_score.grid(axis="y", alpha=0.3)
ax_score.legend()

ax_time.set_xticks(x, AGENTS)
ax_time.set_yscale("log")
ax_time.set_ylabel("Thời gian mỗi nước (ms, thang log)")
ax_time.set_title("Chi phí suy luận trung bình và độ trễ P95")
ax_time.grid(axis="y", alpha=0.3)
ax_time.legend()

fig.tight_layout()
out_perf = FIG_DIR / "score_and_latency.png"
fig.savefig(out_perf, dpi=160, bbox_inches="tight")
print("Đã lưu:", out_perf)
plt.show()

Đã lưu: C:\Users\Admin\AdvancedAi_for_CARO\notebooks\figures\score_and_latency.png


In [4]:
primary_mode = modes[0]
primary_recs = runs[primary_mode]
adv = side_advantage(primary_recs)
lengths = [len(r["moves"]) for r in primary_recs]

fig, (ax_side, ax_len) = plt.subplots(1, 2, figsize=(11.5, 4.3))

labels = ["X thắng (đi trước)", "O thắng (đi sau)", "Hòa"]
counts = [adv["x_wins"], adv["o_wins"], adv["draws"]]
bars = ax_side.bar(labels, counts, color=["#2b7bba", "#e06d53", "#8c8c8c"])
for b, c in zip(bars, counts):
    pct = (c / adv["games"] * 100.0) if adv["games"] else 0.0
    ax_side.text(b.get_x() + b.get_width() / 2, c + max(1, adv["games"] * 0.01), f"{c} ({pct:.1f}%)", ha="center")
ax_side.set_ylim(0, max(counts) * 1.18 + 1)
ax_side.set_ylabel("Số ván")
ax_side.set_title(f"Tác động của lượt đi trước ({primary_mode}, N={adv['games']})")

ax_len.hist(lengths, bins=range(min(lengths), max(lengths) + 2), edgecolor="black", alpha=0.8)
ax_len.axvline(np.mean(lengths), color="red", linestyle="--", label=f"TB = {np.mean(lengths):.1f} nước")
ax_len.set_xlabel("Tổng số nước đi mỗi ván")
ax_len.set_ylabel("Số ván")
ax_len.set_title("Phân bố độ dài ván đấu trên bàn 9×9")
ax_len.legend()

fig.tight_layout()
out_side = FIG_DIR / "side_advantage_and_length.png"
fig.savefig(out_side, dpi=160, bbox_inches="tight")
print("Đã lưu:", out_side)
plt.show()

Đã lưu: C:\Users\Admin\AdvancedAi_for_CARO\notebooks\figures\side_advantage_and_length.png


## Ghi chú mở rộng cho giai đoạn tích hợp Level 5–7 và Ablation

- Khi Thành viên 1 hoàn tất `feat/ablation-search` và `feat/cnn`, bổ sung tên `'cnn'` vào danh sách `AGENTS` ở trên và đọc thêm file log huấn luyện/ablation từ `logs/` để xuất đường cong loss/accuracy cùng biểu đồ so sánh bật/tắt cắt tỉa Alpha-Beta.
- Khi Thành viên 2 hoàn tất `feat/mcts-time`, chạy lại lệnh `arena.round_robin` với các mức `simulations ∈ {100, 400, 1000, 2000}` để vẽ đường cong độ nhạy của MCTS/UCT.